# 04 - Modelo, escenario de coste FP = FN = 10

**Objetivo:** Entrenar y optimizar el modelo de concesion de credito para el escenario de
coste amplificado (FP=FN=10), generando las predicciones de produccion `cs_produccion2.csv`.

**Entregable(s) que produce este notebook:**
- `results/models/modelo_coste10.joblib`
- fila `coste10` en `results/tables/umbrales_coste.csv`
- `results/predicciones/cs_produccion2.csv` (entregable oficial)

## Conectores

**Que recibe de los notebooks anteriores:**
- `data/processed/train.parquet`
- `data/processed/test.parquet`
- `data/processed/produccion.parquet`
- `results/models/preprocessing_pipeline.joblib` (de `02_preprocesado.ipynb`)

**Que entrega a los notebooks siguientes:**
- `results/models/modelo_coste10.joblib`
- `results/tables/umbrales_coste.csv` (fila `coste10`)
- `results/predicciones/cs_produccion2.csv`

consumidos por `05_auditoria_subrogado.ipynb`, `06_contrafactuals.ipynb`, `07_shap.ipynb` y
por `99_ENTREGA.ipynb`.

## Decisiones que afectan a este notebook

- **D-0.1** (modelo supervisado vs Multiarmed Bandit, ABIERTA en `docs/DECISIONES.md`): este
  notebook asume, como `03_modelo_coste1.ipynb`, la rama de clasificador supervisado con ajuste
  de umbral posterior; si D-0.1 se resuelve hacia Multiarmed Bandit, el planteamiento de este
  notebook (entrenar + barrer umbral) quedaria obsoleto y habria que rehacerlo sobre la logica
  de recompensa/coste online.
- **D-0.2** (un solo modelo con dos umbrales/politicas de decision, o dos modelos distintos para
  coste 1 y coste 10, ABIERTA): es la decision mas directamente relevante para este notebook. Si
  se resuelve hacia "un modelo, dos umbrales", la seccion 2 se limita a cargar
  `results/models/modelo_coste1.joblib` y reutilizar sus probabilidades; si se resuelve hacia
  "dos modelos", la seccion 2 reentrena un modelo especifico para este escenario.
  `docs/teoria/cost_sensitive.md` (secciones 2.4 y 3.5) apunta, a partir del patron de los
  notebooks de partida, hacia la primera opcion (un modelo, barrido de umbral), pero la decision
  sigue abierta.
- **D-0.3** (familia de modelo: arboles boosted vs red neuronal vs lineal, ABIERTA): este
  notebook debe comprobar empiricamente si el umbral optimo cambia sustancialmente al
  multiplicar el coste por 10 respecto al umbral hallado en `03_modelo_coste1.ipynb`. Segun
  `docs/teoria/cost_sensitive.md` (secciones 2.3 y 3.4), como ambos escenarios oficiales del
  taller son simetricos dentro de si mismos (C_FP=C_FN=1 y, por separado, C_FP=C_FN=10), la
  prediccion teorica es que el umbral optimo **no** deberia desplazarse entre escenarios, solo
  deberia escalarse x10 el coste esperado. Este notebook debe verificar si esa prediccion se
  cumple sobre el dataset real de credito y documentar cualquier discrepancia.
- **Propuesta D-3.1** (compartida con `03_modelo_coste1.ipynb`, donde se introduce): metodo de
  busqueda del umbral de decision: (a) barrido exhaustivo sobre los valores unicos de
  `predict_proba` en el conjunto de test, evaluando el coste esperado con la matriz FP=FN=10 en
  cada punto (apoyado en `src/cost_utils.py`, patron empirico de `docs/teoria/cost_sensitive.md`
  seccion 2.2), o (b) solucion analitica cerrada `theta* = C_FP / (C_FP + C_FN)`, que aqui
  (C_FP=C_FN=10) tambien da `theta*=0.5` -igual que en el escenario 1- porque la formula solo
  depende del cociente entre costes, no de su escala conjunta (ver `docs/teoria/cost_sensitive.md`
  seccion 2.3). Este notebook debe usar el mismo metodo que se fije en `03_modelo_coste1.ipynb`
  para que ambos escenarios sean comparables; (a) sirve tambien como verificacion empirica de (b).
- **Propuesta D-4.1**: si el escenario de coste=10 exige revisar la familia de modelo elegida en
  D-0.3 por una mayor necesidad de conservadurismo (por ejemplo, un modelo con probabilidades
  mejor calibradas en la cola alta de riesgo, dado que aqui un Falso Negativo pesa 10x mas).
  Pendiente de evidencia empirica: si D-0.3 ya se cierra con un modelo bien calibrado, no deberia
  hacer falta cambiar de familia solo por escalar el coste; este notebook debe registrar si en la
  practica aparece algun indicio de lo contrario (por ejemplo, mal comportamiento del barrido de
  umbral en el extremo superior de probabilidad).

## 1. Carga de datos preprocesados y del pipeline

Carga de `data/processed/train.parquet`, `data/processed/test.parquet`, `data/processed/produccion.parquet` y de `results/models/preprocessing_pipeline.joblib` (generados en `02_preprocesado.ipynb`).

In [ ]:
# TODO: 1. Carga de datos preprocesados y del pipeline

## 2. Reutilizacion o reentrenamiento del modelo segun D-0.2

Segun como se resuelva D-0.2: reutilizar el modelo ya entrenado en `03_modelo_coste1.ipynb` (`results/models/modelo_coste1.joblib`) si la decision es "un modelo, dos umbrales", o entrenar un modelo nuevo especifico para este escenario (familia fijada en D-0.3) si la decision es "dos modelos".

In [ ]:
# TODO: 2. Reutilizacion o reentrenamiento del modelo segun D-0.2

## 3. Calculo del coste esperado (cost_utils, matriz FP=FN=10)

Calculo del coste esperado sobre el conjunto de test usando la funcion comun de `src/cost_utils.py` (ver propuesta D-3.1), instanciada con la matriz de coste C_FP=C_FN=10.

In [ ]:
# TODO: 3. Calculo del coste esperado (cost_utils, matriz FP=FN=10)

## 4. Optimizacion del umbral de decision

Barrer el umbral de decision sobre las probabilidades (`predict_proba`) del conjunto de test, buscando el que minimiza el coste esperado bajo la matriz de coste=10, y comparar ese optimo con el hallado en `03_modelo_coste1.ipynb` (ver D-0.3).

In [ ]:
# TODO: 4. Optimizacion del umbral de decision

## 5. Evaluacion en el conjunto de test

Evaluar el modelo con el umbral optimo elegido (metricas de clasificacion + coste esperado final) y anadir la fila `coste10` a `results/tables/umbrales_coste.csv`.

In [ ]:
# TODO: 5. Evaluacion en el conjunto de test

## 6. Generacion de predicciones de produccion (cs_produccion2.csv)

Aplicar el modelo y el umbral optimo de este escenario sobre `data/processed/produccion.parquet` y generar `results/predicciones/cs_produccion2.csv` (entregable oficial).

In [ ]:
# TODO: 6. Generacion de predicciones de produccion (cs_produccion2.csv)

## 7. Conclusiones del escenario y comparacion con coste1

Comparar el umbral optimo y el coste esperado de este escenario (FP=FN=10) frente al escenario FP=FN=1 de `03_modelo_coste1.ipynb`, contrastando con la prediccion teorica de `docs/teoria/cost_sensitive.md` (seccion 3.4): que el umbral optimo no deberia desplazarse y el coste esperado deberia escalar aproximadamente x10.

In [ ]:
# TODO: 7. Conclusiones del escenario y comparacion con coste1